In [2]:
import pandas as pd
import sys; sys.path.append("..")
from src.config import RESULTS_DIR, MODEL_NAME, RANDOM_SEED
from src.eval.fairness import choose_threshold, group_metrics, fairness_gaps

In [3]:
run = f"{MODEL_NAME}_seed{RANDOM_SEED}"
pred_dir = RESULTS_DIR / "predictions"

In [4]:

val  = pd.read_csv(pred_dir / f"{run}_fitz_val.csv")
test = pd.read_csv(pred_dir / f"{run}_fitz_test.csv")
ddi  = pd.read_csv(pred_dir / f"{run}_ddi.csv")

print(val.shape, test.shape, ddi.shape)

(648, 5) (648, 5) (656, 5)


In [5]:
THRESHOLD = choose_threshold(val, target_sensitivity=0.90)
print(f"frozen threshold: {THRESHOLD:.4f}")

frozen threshold: 0.0937


In [6]:
for name, df in [("fitz_val", val), ("fitz_test", test), ("ddi", ddi)]:
    all_auc = group_metrics(df, THRESHOLD).loc["all", "auc"]
    print(f"{name:10s} all AUC = {all_auc:.4f}")

fitz_val   all AUC = 0.8949
fitz_test  all AUC = 0.8716
ddi        all AUC = 0.6680


In [7]:
for name, df in [("fitz_test", test), ("ddi", ddi)]:
    print(f"\n=== {name} (threshold={THRESHOLD:.3f}) ===")
    print(group_metrics(df, THRESHOLD).round(3))


=== fitz_test (threshold=0.094) ===
              n  n_malignant    auc  sensitivity  specificity
skin_group                                                   
light       346          179  0.835        0.894        0.599
medium      240          114  0.929        0.930        0.802
dark         62           31  0.824        0.774        0.677
all         648          324  0.872        0.895        0.685

=== ddi (threshold=0.094) ===
              n  n_malignant    auc  sensitivity  specificity
skin_group                                                   
light       208           49  0.681        0.878        0.365
medium      241           74  0.665        0.838        0.449
dark        207           48  0.658        0.708        0.497
all         656          171  0.668        0.813        0.437


In [8]:
for name, df in [("fitz_test", test), ("ddi", ddi)]:
    g = fairness_gaps(df, THRESHOLD)
    print(f"\n{name}:")
    for k, v in g.items():
        print(f"  {k:38s} {v:+.4f}")


fitz_test:
  auc_gap_light_minus_dark               +0.0106
  sensitivity_gap_light_minus_dark       +0.1197
  equalized_odds_difference              +0.2028

ddi:
  auc_gap_light_minus_dark               +0.0224
  sensitivity_gap_light_minus_dark       +0.1692
  equalized_odds_difference              +0.1692


In [9]:
from fairlearn.metrics import equalized_odds_difference

for name, df in [("fitz_test", test), ("ddi", ddi)]:
    ours = fairness_gaps(df, THRESHOLD)["equalized_odds_difference"]
    theirs = equalized_odds_difference(
        df["target"], (df["prob"] >= THRESHOLD).astype(int),
        sensitive_features=df["skin_group"],
    )
    print(f"{name:10s} ours={ours:.4f}  fairlearn={theirs:.4f}  "
          f"diff={abs(ours - theirs):.2e}")

fitz_test  ours=0.2028  fairlearn=0.2028  diff=8.33e-17
ddi        ours=0.1692  fairlearn=0.1692  diff=0.00e+00
